# 04 — DeepSeek-web Probe (DeeperSeek)
Tests free chat.deepseek.com as gap-step model. Expect fragility — Qwen stays default.
Needs: `pip install DeeperSeek`, env `DEEPSEEK_TOKEN` (login once in browser, copy token) or email/password.

In [6]:
import os
print('token set:', bool(os.environ.get('DEEPSEEK_TOKEN') or os.environ.get('DEEPSEEK_EMAIL')))

token set: True


In [2]:
from DeeperSeek import DeepSeek

In [3]:

import inspect
print([m for m in dir(DeepSeek) if not m.startswith('_')][:20])

['delete_chats', 'initialize', 'logout', 'regenerate_response', 'reset_chat', 'retrieve_token', 'send_message', 'switch_account', 'switch_chat', 'switch_theme']


In [4]:
LIMS="KD open-set degrades; SLM loses numerics; OpenSet needs labels; rules fail rare; synthetic breaks semantics; LLM not ECU-deployable"
PROMPT=f"Thesis: deep learning for in-vehicle CAN intrusion detection\nLimitations:\n{LIMS}\n\nGive: 2 blind spots, 1 cross-pollination (Method X from A -> Problem Y from B), 2 testable hypotheses with dataset + metric. Be concrete."
# Fill per DeeperSeek docs (token or email login), then send PROMPT. Paste reply back.
print(PROMPT)

Thesis: deep learning for in-vehicle CAN intrusion detection
Limitations:
KD open-set degrades; SLM loses numerics; OpenSet needs labels; rules fail rare; synthetic breaks semantics; LLM not ECU-deployable

Give: 2 blind spots, 1 cross-pollination (Method X from A -> Problem Y from B), 2 testable hypotheses with dataset + metric. Be concrete.


In [5]:
os.getenv("DEEPSEEK_TOKEN")

In [6]:
api = DeepSeek(
    token=os.getenv("DEEPSEEK_TOKEN"),
    email=os.getenv("DEEPSEEK_EMAIL"),
    password=os.getenv("DEEPSEEK_PASSWORD"),
    headless=True
)

In [7]:
await api.initialize()


Timeout: Cloudflare challenge elements not found or not visible within 15 seconds.


TimeoutError: Timeout (10s) waiting for element with selector: 'div[class="ds-checkbox ds-checkbox--none ds-checkbox--bordered"]'

In [7]:
import zendriver
from bs4 import BeautifulSoup
b = await zendriver.start(headless=True)
await b.get('https://chat.deepseek.com/')
try:
    await b.main_tab.verify_cf()
except Exception as e:
    print('cf:', type(e).__name__)
html = await b.main_tab.evaluate('document.documentElement.outerHTML', await_promise=True, return_by_value=True)
soup = BeautifulSoup(html, 'html.parser')
print('== inputs ==')
for i in soup.find_all('input'):
    print(i.get('type'), '|', (i.get('class') or [])[:4], '|', (i.get('placeholder') or '')[:40])
print('== buttons/role=button (first 15) ==')
n = 0
for el in soup.find_all(attrs={'role': 'button'}):
    print((el.get('class') or [])[:4], '|', el.get_text(' ', strip=True)[:40])
    n += 1
    if n >= 15:
        break
print('== checkbox-ish divs ==')
for el in soup.find_all('div', class_=lambda c: c and 'check' in ' '.join(c).lower()):
    print(el.get('class'), '|', el.get_text(' ', strip=True)[:60])
print('== textareas ==')
for el in soup.find_all('textarea'):
    print(el.get('class'), '|', (el.get('placeholder') or '')[:40])

Timeout: Cloudflare challenge elements not found or not visible within 15 seconds.


cf: TimeoutError
== inputs ==
text | ['ds-input__input'] | Phone number / email address
password | ['ds-input__input'] | Password
== buttons/role=button (first 15) ==
['ds-button', 'ds-button--iconLabelPrimary', 'ds-button--icon', 'ds-button--capsule'] | 
['ds-button', 'ds-button--textInheritedPrimary', 'ds-button--text', 'ds-button--capsule'] | Forgot password?
['ds-button', 'ds-button--textInheritedPrimary', 'ds-button--text', 'ds-button--capsule'] | Sign up
['ds-button', 'ds-button--primary', 'ds-button--filled', 'ds-button--capsule'] | Log in
['ds-button', 'ds-button--textLabelTertiary', 'ds-button--text', 'ds-button--capsule'] | Log in with Google
['ds-button', 'ds-button--textLabelTertiary', 'ds-button--text', 'ds-button--capsule'] | Login with Apple
== checkbox-ish divs ==
== textareas ==


In [8]:
import json as _json
tok = os.getenv('DEEPSEEK_TOKEN')
assert tok, 'export DEEPSEEK_TOKEN first'
await b.main_tab.evaluate(f"localStorage.setItem('userToken', JSON.stringify({{value: '{tok}', __version: '0'}}))", await_promise=True, return_by_value=True)
await b.main_tab.reload()
import asyncio as _a
await _a.sleep(4)
html2 = await b.main_tab.evaluate('document.documentElement.outerHTML', await_promise=True, return_by_value=True)
s2 = BeautifulSoup(html2, 'html.parser')
print('== textareas (chat box) ==')
for el in s2.find_all('textarea'):
    print(el.get('class'), '|', (el.get('placeholder') or '')[:60])
print('== send-ish buttons ==')
n = 0
for el in s2.find_all(attrs={'role': 'button'}):
    t = el.get_text(' ', strip=True)[:40]
    print((el.get('class') or [])[:5], '|', t)
    n += 1
    if n >= 20:
        break
print('== ds-markdown present:', bool(s2.find(class_=lambda c: c and 'ds-markdown' in c)), '==')

== textareas (chat box) ==
['_27c9245', 'ds-scroll-area', 'ds-scroll-area--show-on-focus-within', 'ds-scroll-area--enabled', 'd96f2d2a'] | Message DeepSeek
== send-ish buttons ==
['ds-button', 'ds-button--iconLabelPrimary', 'ds-button--icon', 'ds-button--capsule', 'ds-button--m'] | 
['ds-button', 'ds-button--iconLabelPrimary', 'ds-button--icon', 'ds-button--capsule', 'ds-button--m'] | 
['ds-button', 'ds-button--iconLabelPrimary', 'ds-button--icon', 'ds-button--capsule', 'ds-button--m'] | 
['ds-button', 'ds-button--iconLabelTertiary', 'ds-button--icon', 'ds-button--capsule', 'ds-button--m'] | 
['ds-button', 'ds-button--iconLabelTertiary', 'ds-button--icon', 'ds-button--capsule', 'ds-button--m'] | 
['ds-button', 'ds-button--iconLabelTertiary', 'ds-button--icon', 'ds-button--capsule', 'ds-button--m'] | 
['ds-button', 'ds-button--iconLabelTertiary', 'ds-button--icon', 'ds-button--capsule', 'ds-button--xs'] | 
['ds-button', 'ds-button--iconLabelTertiary', 'ds-button--icon', 'ds-button--caps

In [9]:
print('== buttons: aria-label / data-testid / title ==')
seen = set()
for el in s2.find_all(attrs={'role': 'button'}):
    key = (el.get('aria-label'), el.get('data-testid'), el.get('title'), tuple((el.get('class') or [])[:3]))
    if key in seen:
        continue
    seen.add(key)
    print(key)
print('== textarea parent chain (3 up) ==')
ta = s2.find('textarea')
p = ta
for _ in range(4):
    p = p.parent
    print(p.name, (p.get('class') or [])[:4])

== buttons: aria-label / data-testid / title ==
(None, None, None, ('ds-button', 'ds-button--iconLabelPrimary', 'ds-button--icon'))
(None, None, None, ('ds-button', 'ds-button--iconLabelTertiary', 'ds-button--icon'))
(None, None, None, ('ds-button', 'ds-button--iconLabelSecondary', 'ds-button--icon'))
(None, None, None, ('ds-button', 'ds-button--primary', 'ds-button--filled'))
== textarea parent chain (3 up) ==
div ['_24fad49']
div ['_020ab5b']
div ['_77cefa5', '_9996a53']
div ['aaff8b8f']
